# 02b/(선택) 배포 전 로컬에서 미리 띄워보기: 텍스트 분류(intent)

**요약**: 학습한 모델을 **내 GPU에서 먼저 서빙해 보고** 클라우드로 넘어갑니다. 클라우드와 같은 엔진(vLLM)을 쓰므로, 여기서 응답이 나오면 배포도 거의 그대로 됩니다.

**목적**: SageMaker에 배포하면 GPU를 새로 띄우고 컨테이너를 내려받느라 **한 번에 5~15분**이 걸립니다. 그런데 서빙이 안 되는 이유는 대개 모델 파일 문제라, 로컬에서 30초면 같은 오류를 볼 수 있습니다.

**배경**: 배포를 눌러 놓고 10분 기다렸다가 실패를 확인하고, 고쳐서 또 10분 기다리는 일이 반복됩니다. 로컬에서 미리 확인하면 이 왕복을 없앨 수 있습니다.

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

## 이 노트북을 실행하려면
- **로컬 GPU가 필요합니다.** GPU가 없는 환경이면 이 노트북을 건너뛰고 `03_deploy_endpoint`로 바로 가세요(선택 단계입니다).
- **vLLM 설치**: `uv pip install 'vllm>=0.25'`. 클라우드에서 쓰는 버전과 맞추면 결과가 더 잘 재현됩니다(`.env`의 `VLLM_IMAGE_URI` 태그를 보세요: 실측 0.25.1 / 0.26.0).
- **모델 파일**: 아래 §1이 자동으로 준비합니다(로컬 학습 결과 또는 S3 학습 산출물).

**서버는 터미널에서, 호출은 노트북에서** 합니다. vLLM 서버는 Ctrl-C까지 계속 실행되므로 노트북 셀에 넣으면 그 셀이 끝나지 않아 커널이 멈춘 것처럼 보입니다.

> 이 kit의 클라우드 서빙 경로는 **vLLM(기본) / SGLang / DJL LMI** 셋인데 모두 vLLM 계열 엔진입니다. 그래서 로컬 확인도 `vllm serve` 하나로 통일했습니다: 여기서 뜨면 세 경로 모두 통과할 가능성이 높습니다.

## 1. 모델 파일 준비 (자동)
서빙할 모델 폴더(`config.json` + 가중치)를 아래 셀이 알아서 준비합니다:
- 로컬 학습 결과(`./out`)가 있으면 그걸 쓰고,
- 없으면 `%store`에 저장된 `model_data`(SageMaker 학습 산출물)를 S3에서 내려받아 풉니다.

SageMaker로 학습했다면 로컬 `out`이 없는 게 정상입니다: 이때 S3에서 자동으로 받습니다(수 GB라 처음 한 번은 몇 분 걸립니다).
**재학습한 뒤라면**: 이전에 받아 둔 파일이 있어도 산출물이 바뀌면 **자동으로 다시 내려받습니다**(안 그러면 옛 모델을 검증하게 됩니다: 실제로 겪은 함정입니다).
> SFT만 했든 SFT→GRPO까지 했든 확인 방법은 같습니다. 둘 다 같은 형식의 텍스트 모델로 저장되고, `model_data`가 가리키는 것이 곧 확인 대상입니다.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
from common import config, aws_utils, model_inspect as mi

# %store의 model_data(학습 산출물 S3 URI)를 읽어 옵니다.
try:
    get_ipython().run_line_magic('store', '-r model_data')
except Exception:
    pass

# 리전 가드: %store 값은 리전을 바꿔도 남으므로 옛 리전 버킷을 가리킬 수 있습니다.
model_data = aws_utils.ensure_model_data_in_region(
    locals().get('model_data'), config.AWS_REGION, job_prefix='gemma-classification-train')

# 로컬 모델 디렉토리 확보: 'out'(로컬 dry-run)이 있으면 그것을, 없으면 S3 아티팩트를 해제.
#    아티팩트가 바뀌면 자동으로 다시 내려받습니다(캐시 스탬프 비교) → 재학습 후에도 옛 모델을
#    검증하는 실수를 막습니다. 강제로 다시 받으려면 force=True.
MODEL_DIR = mi.prepare_local_model(model_data, config.AWS_REGION)

# vLLM으로 뜰 수 있는지 '실제 체크포인트 텐서 키'로 판정합니다(config 값만으론 알 수 없음).
#    구현과 배경 설명: common/model_inspect.py
info = mi.inspect_servability(MODEL_DIR)
ENGINE, is_text_only = info['engine'], info['is_text_only']

## 2. vLLM 서버 띄우기 (터미널에서)
`vllm serve`는 **OpenAI 호환 서버**를 :8000에 띄웁니다. 즉 OpenAI SDK나 `curl`로 그대로 호출할 수 있고, 클라우드에 배포한 뒤에도 같은 형식으로 부르게 됩니다.

아래 셀이 복사해 붙일 명령을 만들어 줍니다. **새 터미널**을 열어(VS Code: Terminal → New Terminal) 이 트랙 폴더에서 실행하세요.
- 일반(텍스트 모델): `bash scripts/serve_local_vllm.sh <MODEL_DIR>`
- 멀티모달 모델을 **텍스트로만** 쓰려면: `TEXT_ONLY=1 bash scripts/serve_local_vllm.sh <MODEL_DIR>`
- 멀티모달로(이미지 입력 허용): `MULTIMODAL=1 bash scripts/serve_local_vllm.sh <MODEL_DIR>`

터미널에 `Uvicorn running on http://0.0.0.0:8000`이 보이면 준비된 것입니다(모델 로딩까지 수십 초~수 분). **확인이 끝나면 Ctrl-C로 꼭 종료하세요**: GPU를 계속 점유합니다.

In [ ]:
# 위 셀의 MODEL_DIR을 그대로 쓰는 명령을 출력합니다(복사해서 터미널에 붙여넣기).
if ENGINE != 'vllm':
    print('KV-shared 텐서가 누락된 체크포인트입니다: 최신 train.py로 re-export 후 다시 시도하세요.')
else:
    flag = '' if is_text_only else 'TEXT_ONLY=1 '
    print(f"{flag}bash scripts/serve_local_vllm.sh {MODEL_DIR}")

## 3. 호출해 보기: OpenAI 호환 API
서버가 떴으면 실제로 불러 봅니다. **부르는 방법이 세 가지**인데 전부 같은 API를 씁니다: 익숙한 것을 고르세요.

| 방법 | 언제 쓰나 |
|---|---|
| **`requests`** (아래 3-A) | 의존성 없이 바로. 응답 원본(JSON)을 그대로 보고 싶을 때 |
| **OpenAI SDK** (3-B) | 기존 OpenAI 코드를 그대로 재사용. `base_url`만 바꿉니다 |
| **`curl`** (3-C) | 터미널에서 빠르게 확인 |

프롬프트는 `messages`로 보냅니다: **chat template을 서버가 적용**하므로 우리가 조립할 필요가 없습니다. raw 문자열을 보내면 template이 빠져 반복/저품질 출력이 납니다(실측).

**학습에 쓴 것과 같은 형태로 물어야 합니다.** `track_data.SYSTEM_PROMPT`를 함께 보내고, 입력도 학습 데이터와 같은 구조로 줍니다. 이걸 빼면 파인튜닝한 모델이 **일반 챗봇처럼 답해** 학습이 안 된 것처럼 보입니다: 아래 §3-D에서 같은 모델의 응답이 어떻게 달라지는지 직접 비교합니다.

### 3-A. `requests` (기본)

In [ ]:
assert ENGINE == 'vllm', ('KV-shared 텐서가 누락된 체크포인트라 vLLM으로 뜨지 않습니다. '
                          '최신 train.py로 re-export 후 다시 시도하세요(자동 복원).')
import requests, json
import importlib, track_data as td; importlib.reload(td)   # 학습에 쓴 SYSTEM_PROMPT 재사용
BASE = 'http://localhost:8000/v1'
try:
    served = requests.get(f'{BASE}/models', timeout=5).json()['data'][0]['id']
    print('server up. served model:', served)
except Exception as e:
    raise SystemExit(f'vLLM 서버에 연결 실패({e}). 위 §2 명령으로 서버를 먼저 띄우세요.')

user = "My new card hasn't arrived yet, what should I do?"
messages = [{'role': 'system', 'content': td.SYSTEM_PROMPT},
            {'role': 'user', 'content': user}]
resp = requests.post(f'{BASE}/chat/completions', timeout=120, json={
    'model': served, 'messages': messages,
    'max_tokens': 256, 'temperature': 0.2,
}).json()
print('--- system ---\n', td.SYSTEM_PROMPT)
print('--- user ---\n', user)
print('--- SLM output ---\n', resp['choices'][0]['message']['content'])
print('--- usage ---\n', resp.get('usage'))   # 토큰 사용량(프롬프트/생성)

### 3-B. OpenAI SDK: `base_url`만 바꿔서 그대로
OpenAI 코드를 이미 쓰고 있다면 **`base_url`과 `api_key`만 바꾸면** 우리 모델을 부를 수 있습니다(vLLM은 인증을 요구하지 않으므로 `api_key`는 아무 값이나 넣습니다). 스트리밍도 그대로 됩니다.

In [ ]:
# OpenAI SDK (없으면 설치)
import shutil, subprocess, sys
_pkgs = ['openai>=1.100.0']
if shutil.which('uv'):
    subprocess.run(['uv', 'pip', 'install', '--python', sys.executable, '-q', *_pkgs], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs], check=True)
print('installed:', _pkgs)
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')  # 로컬 vLLM은 인증 없음

# §3-A에서 만든 messages를 그대로 씁니다(system prompt + 학습 형태 입력).
#    여기서 system을 빼고 맨 질문만 보내면 일반 챗봇처럼 답합니다: §3-D에서 비교합니다.
out = client.chat.completions.create(
    model=served,
    messages=messages,
    max_tokens=256, temperature=0.2,
)
print('--- SLM output ---\n', out.choices[0].message.content)

# 스트리밍: 토큰이 생성되는 대로 받아 첫 응답 체감을 줄입니다(vLLM은 native 지원).
print('\n--- streaming ---')
for chunk in client.chat.completions.create(
        model=served, messages=messages,
        max_tokens=256, temperature=0.2, stream=True):
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

### 3-C. `curl`: 터미널에서 한 줄로
payload를 **파일로 저장한 뒤 `-d @파일`** 로 넘깁니다. 프롬프트에 `'`(아포스트로피)나 한글이 들어가면 shell 인용이 깨지기 때문입니다(`What's...` 같은 문장에서 바로 발생).

In [ ]:
# payload를 파일로 저장(따옴표 문제 회피) → 출력된 curl을 복사해 터미널에 붙여넣으세요.
import json as _json
# §3-A의 messages를 그대로 씁니다(system prompt 포함).
_payload = {'model': served, 'messages': messages, 'max_tokens': 256, 'temperature': 0.2}
with open('req.json', 'w', encoding='utf-8') as f:
    _json.dump(_payload, f, ensure_ascii=False)
print('saved: req.json')
print()
print('curl -s http://localhost:8000/v1/chat/completions \\')
print("  -H 'Content-Type: application/json' \\")
print("  -d @req.json | jq -r '.choices[0].message.content'")
print()
print('# jq가 없으면: | python3 -c \'import sys,json; print(json.load(sys.stdin)[\"choices\"][0][\"message\"][\"content\"])\'')

### 3-D. 프롬프트를 어떻게 주느냐가 결과를 바꿉니다
**같은 모델**에 같은 질문을 세 가지 방식으로 물어 응답을 비교합니다. 파인튜닝이 잘 됐는지 판단할 때 가장 먼저 확인해야 할 부분입니다: 학습과 다른 형태로 물으면 학습 효과가 안 보입니다.

실측 예(추출 트랙):
```
A) system 없음, 스키마 없음 → "I do not have real-time access to weather..."   (일반 챗봇)
B) system 있음, 스키마 없음 → {"name": "get_current_weather", ...}            (함수명 추측)
C) system + 스키마         → {"name": "get_weather", "arguments": {...}}      정확
```
C가 학습 데이터와 같은 형태입니다. **배포 후 호출부(04 평가/05 agentic)도 모두 C 방식**으로 보냅니다.

In [ ]:
# 같은 모델/같은 질문/다른 프롬프트 구성 → 응답이 어떻게 달라지는지 확인합니다.
def _ask(msgs):
    r = requests.post(f'{BASE}/chat/completions', timeout=120, json={
        'model': served, 'messages': msgs, 'max_tokens': 200, 'temperature': 0.2}).json()
    return r['choices'][0]['message']['content'].strip()

bare = "My new card hasn't arrived yet, what should I do?"   # 학습 형태가 아닌 '맨' 질문

print('=' * 70)
print('A) system 없음 + 학습 형태 아님  ← 이렇게 물으면 학습 효과가 안 보입니다')
print('-' * 70)
print(_ask([{'role': 'user', 'content': bare}])[:300])
print()
print('=' * 70)
print('B) system prompt만 추가')
print('-' * 70)
print(_ask([{'role': 'system', 'content': td.SYSTEM_PROMPT},
            {'role': 'user', 'content': bare}])[:300])
print()
print('=' * 70)
print('C) system + 학습 데이터와 같은 형태  ← 배포 후에도 이 방식으로 호출합니다')
print('-' * 70)
print(_ask(messages)[:300])   # §3-A에서 만든 messages(= serve_example_user)
print('=' * 70)

## 4. (선택) 성능 측정: `vllm bench`
vLLM에는 **벤치마크 CLI가 내장**돼 있습니다(`vllm bench --help`로 확인). 배포 전에 여기서 재 두면 **어떤 인스턴스가 필요한지, 동시 요청을 몇 개까지 받을 수 있는지**를 근거를 갖고 정할 수 있습니다.

| 서브커맨드 | 무엇을 재나 | 서버 필요? |
|---|---|---|
| `bench serve` | **온라인 처리량 + 지연**(TTFT/TPOT/P99): 실제 서빙에 가장 가까움 | 필요 |
| `bench latency` | 배치 1개의 순수 추론 지연 | ❌ 직접 로드 |
| `bench throughput` | 오프라인 일괄 처리량(배치 추론) | ❌ 직접 로드 |
| `bench startup` | 모델 로딩 시간: **endpoint 콜드스타트** 예측에 유용 | ❌ |

**핵심 지표 두 개**
- **TTFT**(Time To First Token): 첫 글자가 나오기까지. 체감 반응 속도를 좌우합니다.
- **TPOT**(Time Per Output Token): 토큰당 생성 시간. 긴 답변의 총 소요를 좌우합니다.

로컬 GPU와 클라우드 인스턴스가 다르면 **절대값은 다릅니다.** 그래도 "동시 요청을 늘리면 어디서 무너지는지", "입력이 길어지면 얼마나 느려지는지" 같은 **경향**은 그대로 쓸 만합니다.

> 아래 셀은 명령만 출력합니다. §2에서 띄운 서버를 그대로 쓰되, **또 다른 터미널**에서 실행하세요.

In [ ]:
# 실행할 명령을 출력합니다(복사해서 **새 터미널**에 붙여넣기: §2의 서버는 그대로 두세요).
#    스크립트가 서버 생존 확인/모델 이름 조회/플래그 조립을 대신 처리합니다.
print(f'cd {os.getcwd()}')
print()
print('# (1) 온라인 서빙: 처음엔 이것만 봐도 충분합니다')
print(f'bash scripts/bench_local_vllm.sh {MODEL_DIR}')
print()
print('# (2) 동시성 한계 찾기: 1→4→8→16 스윕')
print(f'MODE=sweep bash scripts/bench_local_vllm.sh {MODEL_DIR}')
print()
print('# (3) 콜드스타트: endpoint가 InService까지 걸리는 시간을 가늠 (서버 불필요)')
print(f'MODE=startup bash scripts/bench_local_vllm.sh {MODEL_DIR}')
print()
print('# (4) 오프라인 배치 처리량: Batch Transform 규모 산정용 (서버 불필요)')
print(f'MODE=throughput bash scripts/bench_local_vllm.sh {MODEL_DIR}')
print()
print('옵션(env): NUM_PROMPTS=50 CONCURRENCY=8 INPUT_LEN=1024 OUTPUT_LEN=256 SAVE=1')
print('  SAVE=1 을 주면 결과 JSON을 ./bench 에 남깁니다.')

> **측정값을 배포에 어떻게 쓰나**
> - TTFT가 목표보다 크다 → 더 큰 GPU(또는 `--max-model-len`을 줄여 KV 캐시 여유 확보)
> - 동시 8에서 이미 P99가 무너진다 → `03`에서 인스턴스를 키우거나 오토스케일링 최소 인스턴스를 늘립니다
> - 콜드스타트가 길다 → endpoint를 내리지 않고 유지하거나, provisioned 상태를 유지하는 편이 낫습니다

응답이 확인됐다면 서빙 준비가 끝났습니다: **03_deploy_endpoint.ipynb**로 배포하세요(1-A: vLLM/SGLang DLC, 1-B: DJL LMI).

**로컬 vLLM 서버를 Ctrl-C로 종료하세요**: GPU를 계속 점유합니다.

> §1 판정에서 KV-shared 텐서 누락이 나왔다면, 최신 `train.py`로 다시 학습/re-export하면 자동 복원됩니다(`_revive_kv_shared_from_base`가 base에서 그 54개 텐서를 되살려 저장).